# soamp — featurization grid: 4 experiments on Colab GPU

Runs the 2×2 grid of peptide representation (`rdkit_descriptors` |
`peptideclm_embedding`) × organism representation (`vocab_embedding` |
`kmer_composition`), with `attention_fusion_classifier` held fixed so the
comparison isolates the representation rather than the model.

Each cell is a committed config under `config/train/`, executed through
`pipeline/train.py` — the same entrypoint a local run uses. This notebook
adds no training loop of its own; it bootstraps the environment, builds the
one artifact that needs a GPU, and collects results.

| run (`exp_id`) | peptide | organism |
|---|---|---|
| `rdkit_vocab_attnfusion` | RDKit descriptors (13d) | learned embedding |
| `rdkit_kmer_attnfusion` | RDKit descriptors (13d) | genome k-mer (340d) |
| `peptideclm_vocab_attnfusion` | PeptideCLM (768d) | learned embedding |
| `peptideclm_kmer_attnfusion` | PeptideCLM (768d) | genome k-mer (340d) |

All four log to wandb project `soamp`, group `featurization_grid_v1`.

## Before you run

Run `01_smoke_overfit.ipynb` first — it checks every cell constructs and can
drive its loss to zero. A **GPU runtime** matters here only for the PeptideCLM
featurization step; the classifier itself is small.

Set these under **Colab Secrets** (the key icon in the left sidebar), with
notebook access enabled for each:

| Secret | Needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (a fine-grained, read-only PAT is enough) |
| `WANDB_API_KEY` | logging runs to Weights & Biases |
| `WANDB_ENTITY` | optional; your wandb team/username if it isn't your default |

## 1. Environment

In [ ]:
import subprocess
import sys

print("Python:", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU:", gpu)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU: none detected -- set Runtime > Change runtime type > T4 GPU, then rerun.")

In [ ]:
import os

from google.colab import userdata

# Colab Secrets (key icon in the left sidebar), not a committed .env: the repo
# gitignores .env, so nothing sensitive travels with the clone.
# WANDB_API_KEY is required here -- the four runs log to wandb.
REQUIRED = {"GITHUB_TOKEN": True, "WANDB_API_KEY": True, "WANDB_ENTITY": False}

for name, required in REQUIRED.items():
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception as e:
        if required:
            raise RuntimeError(
                f"Colab secret {name!r} is missing. Add it under the key icon "
                f"in the left sidebar and enable notebook access."
            ) from e
        print(f"{name}: not set (optional)")

In [ ]:
import os
import subprocess

REPO = "LukaJinc/soamp"
BRANCH = "main"
WORKDIR = "/content/soamp"

if not os.path.isdir(WORKDIR):
    token = os.environ["GITHUB_TOKEN"]
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         f"https://{token}@github.com/{REPO}.git", WORKDIR],
        capture_output=True, text=True,
    )
    # Scrub the token before anything reaches notebook output, which is saved
    # with the file.
    print((result.stdout + result.stderr).replace(token, "***"))
    if result.returncode != 0:
        raise RuntimeError("git clone failed -- check GITHUB_TOKEN has read access to the repo")
    # Drop the credential from .git/config too, so later git calls can't leak it.
    subprocess.run(
        ["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"],
        check=True,
    )

os.chdir(WORKDIR)
print("HEAD:", subprocess.run(["git", "log", "-1", "--oneline"],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
# requirements-colab.txt first, then the package with --no-deps: installing
# soamp's own pinned dependency set would replace Colab's CUDA-matched torch
# build and drag in curation-only packages this workload never imports.
!pip install -q -r scripts/colab/requirements-colab.txt
!pip install -q -e . --no-deps

In [ ]:
import torch

from soamp.data.factory import build_dataset
from soamp.model.factory import build_model
from soamp.utils.device import resolve_device

device = resolve_device("auto")
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | device: {device}")
if device.type != "cuda":
    print("\nWARNING: running on CPU. The classifier is small enough not to care, but the "
          "PeptideCLM featurization pass will take ~15min instead of seconds.")

## 2. PeptideCLM features (the one GPU-bound step)

Three of the four feature artifacts are committed to the repo and arrive with
the clone — the RDKit descriptors (1.4 MB) and both organism representations
(the k-mer one has its genome vectors baked in, so no FASTAs are needed here).

The fourth, PeptideCLM's 12,371 × 768 embedding matrix, is ~190 MB — past
GitHub's per-file limit — so it is built per-environment. It takes seconds on a
GPU versus ~5 min on CPU. Drive caches it so a later session skips the work.

In [ ]:
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/soamp_cache"
RESULTS_DIR = "/content/drive/MyDrive/soamp_results"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Both are products of the same featurization pass, so they are cached together
# -- a scaler fitted on different embeddings than the ones on disk would be a
# silent correctness bug, not a crash.
PEPTIDECLM_ARTIFACTS = [
    "peptide_features_peptideclm.csv",
    "peptide_feature_scaler_peptideclm.json",
]

In [ ]:
cached = [f for f in PEPTIDECLM_ARTIFACTS if os.path.exists(f"{CACHE_DIR}/{f}")]

if len(cached) == len(PEPTIDECLM_ARTIFACTS):
    for name in PEPTIDECLM_ARTIFACTS:
        shutil.copy(f"{CACHE_DIR}/{name}", f"data/{name}")
        print(f"restored from Drive: {name} ({os.path.getsize(f'data/{name}') / 1e6:.1f} MB)")
else:
    if cached:
        print(f"partial cache ({cached}) -- rebuilding both so they stay consistent")
    print("building PeptideCLM embeddings...")
    !python pipeline/features/01_build_peptide_features.py --config config/features/peptide_peptideclm.yaml
    !python pipeline/features/03_fit_peptide_scaler.py --config config/features/peptide_peptideclm.yaml
    for name in PEPTIDECLM_ARTIFACTS:
        shutil.copy(f"data/{name}", f"{CACHE_DIR}/{name}")
        print(f"cached to Drive: {name}")

## 3. Are all four artifact sets present?

Each grid cell reads its own method-suffixed filenames, so all four coexist in
`data/` without overwriting one another. Checking here turns a missing artifact
into one clear message now rather than a `FileNotFoundError` three runs deep.

In [ ]:
import json

REQUIRED_ARTIFACTS = {
    "shared": [
        "mic_classification_dataset.csv",
        "val_split.json",
    ],
    "peptide: rdkit_descriptors": [
        "peptide_features_rdkit.csv",
        "peptide_feature_scaler_rdkit.json",
    ],
    "peptide: peptideclm_embedding": PEPTIDECLM_ARTIFACTS,
    "organism: vocab_embedding": ["organism_vocab_vocab_embedding.json"],
    "organism: kmer_composition": ["organism_vocab_kmer_composition.json"],
}

missing = []
for group, names in REQUIRED_ARTIFACTS.items():
    for name in names:
        path = f"data/{name}"
        if os.path.exists(path):
            print(f"  {group:34s} {name:42s} {os.path.getsize(path) / 1e6:8.2f} MB")
        else:
            missing.append(f"{group}: {name}")

if missing:
    raise FileNotFoundError("missing artifacts:\n  " + "\n  ".join(missing))

# The two organism artifacts are self-describing -- confirm each really carries
# the method its filename claims, so a stale copy can't silently mislabel a run.
for name, expected in [("organism_vocab_vocab_embedding.json", "vocab_embedding"),
                       ("organism_vocab_kmer_composition.json", "kmer_composition")]:
    actual = json.load(open(f"data/{name}"))["method"]
    assert actual == expected, f"{name} says method={actual!r}, expected {expected!r}"

print("\nall artifacts present and self-consistent")

## 4. Run the grid

Each config is executed as a subprocess, exactly as it would run locally. The
runs are sequential and independent — if one fails, the others still complete
and the failure is reported at the end rather than killing the notebook.

`pipeline/train.py` resolves `device: auto` itself, seeds from config, embeds
the resolved config + git SHA in every checkpoint, and reloads `<exp_id>_best.pth`
before touching the test split — which it does exactly once.

In [ ]:
import subprocess
import time

EXPERIMENTS = [
    "exp_rdkit_vocab",
    "exp_rdkit_kmer",
    "exp_peptideclm_vocab",
    "exp_peptideclm_kmer",
]

run_status = {}
for name in EXPERIMENTS:
    print(f"\n{'=' * 72}\n  {name}\n{'=' * 72}")
    started = time.time()
    result = subprocess.run(
        ["python", "pipeline/train.py", "--config", f"config/train/{name}.yaml"],
        capture_output=True, text=True,
    )
    elapsed = time.time() - started
    # train.py logs through stdlib logging -> stderr; tail it so a failure is
    # readable without scrolling the whole epoch log.
    print("\n".join(result.stderr.strip().splitlines()[-25:]))
    run_status[name] = {"returncode": result.returncode, "seconds": round(elapsed, 1)}
    print(f"\n-> exit {result.returncode} in {elapsed:.0f}s")

failed = [n for n, s in run_status.items() if s["returncode"] != 0]
print(f"\n\n{len(EXPERIMENTS) - len(failed)}/{len(EXPERIMENTS)} runs succeeded")
if failed:
    print(f"FAILED: {failed}")

## 5. Compare the four runs

Pulled back from wandb rather than re-read off local files, so this is the same
record the runs actually published. `test_*` values live in each run's summary,
written after the single held-out test evaluation.

In [ ]:
import pandas as pd
import wandb

PROJECT = "soamp"
GROUP = "featurization_grid_v1"

api = wandb.Api()
entity = os.environ.get("WANDB_ENTITY") or api.default_entity
runs = api.runs(f"{entity}/{PROJECT}", filters={"group": GROUP})

records = []
for run in runs:
    summary, config = run.summary, run.config
    records.append({
        "run": run.name,
        "peptide": config.get("peptide_method"),
        "peptide_dim": config.get("peptide_feature_dim"),
        "organism": config.get("organism_method"),
        "organism_kind": config.get("organism_output_kind"),
        "device": config.get("device"),
        "test_auroc": summary.get("test_auroc"),
        "test_accuracy": summary.get("test_accuracy"),
        "test_f1": summary.get("test_f1"),
        "E. coli": summary.get("test_Escherichia coli_auroc"),
        "S. aureus": summary.get("test_Staphylococcus aureus_auroc"),
        "P. aeruginosa": summary.get("test_Pseudomonas aeruginosa_auroc"),
        "state": run.state,
    })

results_df = pd.DataFrame(records).sort_values("test_auroc", ascending=False)
results_df

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.dropna(subset=["test_auroc"]).set_index("run")
per_organism = plot_df[["E. coli", "S. aureus", "P. aeruginosa"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

plot_df["test_auroc"].sort_values().plot.barh(ax=axes[0], color="#4C78A8")
axes[0].set_xlim(0.5, 1.0)
axes[0].set_xlabel("test AUROC")
axes[0].set_title("Overall (held-out test split)")
axes[0].grid(axis="x", alpha=0.3)

per_organism.plot.bar(ax=axes[1], rot=15, width=0.75)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_ylabel("test AUROC")
axes[1].set_title("Per organism")
axes[1].legend(fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save results off the runtime

Checkpoints and log files live on the Colab VM's disk, which is discarded when
the session ends. wandb already holds the metrics; this keeps the weights.

In [ ]:
import glob

saved = []
for path in glob.glob("reports/checkpoints/*.pth") + glob.glob("reports/train_*_log.txt"):
    destination = f"{RESULTS_DIR}/{os.path.basename(path)}"
    shutil.copy(path, destination)
    saved.append(os.path.basename(path))

results_df.to_csv(f"{RESULTS_DIR}/featurization_grid_results.csv", index=False)
print(f"copied {len(saved)} files + featurization_grid_results.csv to {RESULTS_DIR}")

## Reading the result

Every checkpoint embeds its resolved config, git SHA and seed, so any of these
four runs can be reproduced from the file alone.

Two things this grid does **not** settle:

- **One seed each.** A gap of a point or two of AUROC between neighbouring cells
  is within the noise of a single run. Re-run the interesting cells with a
  different `loop.seed` before reading much into a small difference.
- **Three organisms.** The dataset is scoped to the organisms with curated MIC
  thresholds (*E. coli*, *S. aureus*, *P. aeruginosa*). `kmer_composition` is
  built to generalise to organisms never seen in training — which is exactly
  what three organisms cannot demonstrate. Its payoff should show up as more
  thresholds get filled into `config/thresholds/organism_thresholds.csv`.